# CSV文件读取 `CsvHelper`

## 成员
- `std::ifstream _ifs`：文件输入流，用于读取CSV文件内容
- `char _buffer[1024]`：行数据缓冲区，存储当前读取的行内容
- `std::string _item_splitter`：字段分隔符字符串
- `std::unordered_map<std::string, int32_t> _fields_map`：字段名到列索引的映射表，用于按字段名查找数据
- `std::vector<std::string> _current_cells`：当前行的字段值列表，存储解析后的各个字段值

## 方法
- **核心属性与构造函数**
  - 构造函数: `CsvReader(const char* item_splitter = ",")`
  - 获取字段（列）总数: `inline uint32_t col_count()`
  - 获取所有字段名称（逗号分隔）: `const char* fields() const`
- **文件加载与迭代**
  - 从文件加载并解析表头: `bool load_from_file(const char* filename)`
  - 读取并解析下一行数据: `bool next_row()`
- **按列索引获取数据**
  - 获取32位有符号整数: `int32_t get_int32(int32_t col)`
  - 获取32位无符号整数: `uint32_t get_uint32(int32_t col)`
  - 获取64位有符号整数: `int64_t get_int64(int32_t col)`
  - 获取64位无符号整数: `uint64_t get_uint64(int32_t col)`
  - 获取双精度浮点数: `double get_double(int32_t col)`
  - 获取字符串: `const char* get_string(int32_t col)`
- **按字段名获取数据**
  - 获取32位有符号整数: `int32_t get_int32(const char* field)`
  - 获取32位无符号整数: `uint32_t get_uint32(const char* field)`
  - 获取64位有符号整数: `int64_t get_int64(const char* field)`
  - 获取64位无符号整数: `uint64_t get_uint64(const char* field)`
  - 获取双精度浮点数: `double get_double(const char* field)`
  - 获取字符串: `const char* get_string(const char* field)`

# 日志管理器 `WTSLogger`

# 基础数据管理器 `WTSBaseDataMgr`
管理交易系统运行的基础信息，包括：合约信息、品种信息、交易时间、节假日等。

继承了 `IBaseDataMgr`，参考 [Includes/note.ipynb/数据管理接口层/基础数据管理接口 IBaseDataMgr](../Includes/note.ipynb)

## 成员
- `TradingDayTplMap m_mapTradingDay`：交易日模板映射表，存储各地区的交易日历模板
  - typedef wt_hashmap\<std::string, `TradingDayTpl`\>	TradingDayTplMap;
    - uint32_t _cur_tdate：当前交易日日期，格式为YYYYMMDD
	- `HolidaySet` _holidays：节假日集合，存储所有节假日日期
    	- typedef wt_hashset\<uint32_t\> HolidaySet;
  - 本质上是 **map<地区ID，(当前交易日期，set(节假日ID))>**
- `SessionCodeMap m_mapSessionCode`：时段代码映射表，维护时段与品种的对应关系
  - typedef wt_hashmap\<std::string, `CodeSet`\> SessionCodeMap;
    - typedef fastest_hashset\<std::string\> CodeSet;
  - 本质上是 **map<时段ID，set(品种ID)>**
- `WTSExchgContract* m_mapExchgContract`：按交易所组织的合约信息映射表
  - 本质上是 **map<交易所ID，map\*\<合约ID，合约信息WTSContractInfo\*\>\>**
- `WTSSessionMap* m_mapSessions`：交易时段信息映射表
  - 本质上是 **map<品种ID, 品种交易时段信息WTSSessionInfo\*\>**
- `WTSCommodityMap* m_mapCommodities`：商品品种信息映射表
  - 本质上是 **map<品种ID, 品种信息WTSCommodityInfo\*\>**
- `WTSContractMap* m_mapContracts`：合约信息映射表，支持同名合约的多版本管理
  - 本质上是 **map<合约ID, array(合约信息WTSContractInfo\*\)\>**
  - 因为一个合约可能在多个交易所都有

## 方法

### 核心属性与构造函数

#### 构造函数 
```cpp
WTSBaseDataMgr::WTSBaseDataMgr()
	: m_mapExchgContract(NULL)    // 交易所合约映射表指针初始化为NULL
	, m_mapSessions(NULL)         // 交易时段映射表指针初始化为NULL
	, m_mapCommodities(NULL)      // 商品品种映射表指针初始化为NULL
	, m_mapContracts(NULL)        // 合约映射表指针初始化为NULL
{
	m_mapExchgContract = WTSExchgContract::create();
	m_mapSessions = WTSSessionMap::create();
	m_mapCommodities = WTSCommodityMap::create();
	m_mapContracts = WTSContractMap::create();
}
```

#### 加载交易时段配置 loadSessions
配置文件格式（JSON）示例：
```JSON
{
	"DAY": {
		"name": "日盘",
		"offset": 0,
		"auction": {"from": 85500, "to": 90000},
		"sections": [
				{"from": 90000, "to": 101500},
				{"from": 103000, "to": 150000}
			]
	}
}
```

配置项说明：
- id：时段标识符（如"DAY"、"NIGHT"）
- name：时段显示名称
- offset：时区偏移量（小时）
- auction/auctions：集合竞价时间段
- sections：连续交易时间段数组

```cpp
/**
 * @brief 加载交易时段配置
 * @param filename 交易时段配置文件路径
 * @return bool 加载成功返回true，失败返回false
 * 
 * 该函数从配置文件中加载所有交易时段的定义，是系统初始化的关键步骤之一。
 * 交易时段配置包含了各个品种的交易时间规则，是交易系统正常运行的基础。
 * 

 * 
 * 加载过程：
 * 1. 检查配置文件是否存在
 * 2. 解析JSON格式的配置文件
 * 3. 遍历所有时段定义
 * 4. 创建WTSSessionInfo对象
 * 5. 设置集合竞价时间
 * 6. 添加连续交易时间段
 * 7. 将时段信息加入映射表
 */
bool WTSBaseDataMgr::loadSessions(const char* filename)
{
	// 检查交易时段配置文件是否存在
	if (!StdFile::exists(filename))
	{
		WTSLogger::error("Trading sessions configuration file {} not exists", filename);
		return false;  // 文件不存在，加载失败
	}

	// 使用配置加载器解析配置文件
	// WTSCfgLoader支持JSON、INI等多种格式
	WTSVariant* root = WTSCfgLoader::load_from_file(filename);
	if (root == NULL)
	{
		WTSLogger::error("Loading session config file {} failed", filename);
		return false;  // 配置文件解析失败
	}

	// 遍历配置文件中的所有交易时段定义
	for(const std::string& id : root->memberNames())
	{
		// 获取当前时段的配置对象
		WTSVariant* jVal = root->get(id);

		// 从配置中读取时段的基本信息
		const char* name = jVal->getCString("name");    // 时段显示名称
		int32_t offset = jVal->getInt32("offset");      // 时区偏移量

		// 创建交易时段信息对象
		WTSSessionInfo* sInfo = WTSSessionInfo::create(id.c_str(), name, offset);

		// 处理集合竞价时间配置
		// 支持单个集合竞价时间段（auction）和多个集合竞价时间段（auctions）
		if (jVal->has("auction"))
		{
			// 单个集合竞价时间段配置
			WTSVariant* jAuc = jVal->get("auction");
			sInfo->setAuctionTime(jAuc->getUInt32("from"), jAuc->getUInt32("to"));
		}
		else if (jVal->has("auctions"))
		{
			// 多个集合竞价时间段配置（支持多次集合竞价）
			WTSVariant* jAucs = jVal->get("auctions");
			for (uint32_t i = 0; i < jAucs->size(); i++)
			{
				WTSVariant* jSec = jAucs->get(i);
				sInfo->addAuctionTime(jSec->getUInt32("from"), jSec->getUInt32("to"));
			}
		}

		// 处理连续交易时间段配置
		WTSVariant* jSecs = jVal->get("sections");
		if (jSecs == NULL || !jSecs->isArray())
			continue;  // 没有交易时间段配置，跳过当前时段

		// 遍历所有连续交易时间段
		for (uint32_t i = 0; i < jSecs->size(); i++)
		{
			WTSVariant* jSec = jSecs->get(i);
			// 添加交易时间段（开始时间、结束时间）
			// 时间格式：HHMMSS（如90000表示9:00:00）
			sInfo->addTradingSection(jSec->getUInt32("from"), jSec->getUInt32("to"));
		}

		// 将配置好的交易时段信息添加到映射表中
		m_mapSessions->add(id.c_str(), sInfo);
	}

	// 释放配置文件解析产生的根对象，避免内存泄漏
	root->release();

	// 所有交易时段配置加载成功
	return true;
}
```

#### 加载商品品种配置

#### 加载合约信息配置

#### 加载节假日配置

#### 释放所有资源

### 合约与品种查询接口
    - 按标准ID获取品种信息: `virtual WTSCommodityInfo* getCommodity(const char* stdPID)`
    - 按交易所和品种代码获取品种信息: `virtual WTSCommodityInfo* getCommodity(const char* exchg, const char* pid)`
    - 获取单个合约信息: `virtual WTSContractInfo* getContract(const char* code, const char* exchg = "", uint32_t uDate = 0)`
    - 获取合约列表: `virtual WTSArray* getContracts(const char* exchg = "", uint32_t uDate = 0)`
    - 获取合约数量: `virtual uint32_t getContractSize(const char* exchg = "", uint32_t uDate = 0)`

### 交易时段查询接口
    - 按ID获取交易时段信息: `virtual WTSSessionInfo* getSession(const char* sid)`
    - 按合约代码获取交易时段信息: `virtual WTSSessionInfo* getSessionByCode(const char* code, const char* exchg = "")`
    - 获取所有交易时段信息: `virtual WTSArray* getAllSessions()`
    - 获取使用某时段的品种集合: `CodeSet* getSessionComms(const char* sid)`

### 交易日历与日期计算
    - 判断是否为节假日: `virtual bool isHoliday(const char* stdPID, uint32_t uDate, bool isTpl = false)`
    - 判断是否为交易日: `bool isTradingDate(const char* stdPID, uint32_t uDate, bool isTpl = false)`
    - 根据自然时间计算所属交易日: `virtual uint32_t calcTradingDate(const char* stdPID, uint32_t uDate, uint32_t uTime, bool isSession = false)`
    - 获取当前交易日: `uint32_t getTradingDate(const char* stdPID, uint32_t uOffDate = 0, uint32_t uOffMinute = 0, bool isTpl = false)`
    - 设置当前交易日: `void setTradingDate(const char* stdPID, uint32_t uDate, bool isTpl = false)`
    - 获取下一个交易日: `uint32_t getNextTDate(const char* stdPID, uint32_t uDate, int days = 1, bool isTpl = false)`
    - 获取上一个交易日: `uint32_t getPrevTDate(const char* stdPID, uint32_t uDate, int days = 1, bool isTpl = false)`
    - 获取交易日边界时间（开/收盘）: `virtual uint64_t getBoundaryTime(const char* stdPID, uint32_t tDate, bool isSession = false, bool isStart = true)`